# Critical energy infrastructure in Denmark under attacks


Modern power systems are large-scale, highly interconnected networks where failures can propagate rapidly across regions. Identifying critical components—nodes or transmission elements whose failure leads to severe system imbalance—is essential for ensuring grid reliability and resilience.

This project performs a systematic vulnerability analysis of the Danish transmission grid by combining:

 - Network modeling (graph theory)
 - Optimization-based power flow simulation
 - Scenario-based stress testing
 - Graph Neural Networks (GNNs) for prediction

The objective is to quantify the impact of component failures and identify the most critical nodes and edges under varying system conditions.

Libaries:

In [ ]:
import Evaluation
import cleaning
import pickle
import pandas as pd
import cleaning
import networkx as nx

## Dataset
The dataset is provided by Energinet and contains detailed structural and operational information about the Danish transmission system, including:

 - Substations (Bus sheet)
 - Transmission lines (Line sheet)
 - Generators (Generator sheet)
 - Loads (Load sheet)
 - HVDC interconnectors (HVDC sheet)
 - Transformers (Transformer2 and Transformer3 sheets)

## Data cleaning
The system is modeled as a graph:

Nodes (buses):
- Represent substations
- Attributes include:
    - supply (MW)
    - demand (MW)
    - generation limits (p_min, p_max)
    - energy source (e.g., wind, gas, hydro)
- Cordinates (added manually)

Edges:
- Represent transmission lines and transformers
- Include capacity constraints derived from:
$$
\text { capacity } \approx \sqrt{3} \cdot V \cdot I
$$
- Transformer connections are explicitly included to ensure topological completeness

### Disaggregated vs Aggregated Graphs

The model distinguishes between two graph representations:

#### Disaggregated graph
Multiple nodes per physical bus (one per generation source)
Internal "busbar" edges connect subcomponents
Used to capture generation flexibility and source-level detail
#### Aggregated graph
One node per bus
Aggregates:
- total supply
- total demand
flexibility (p_addable, p_removable)
Used for:
- optimization model
- attack simulations



In [3]:
file_path = "publicdataexportv131450706334_with_lon_lat.xlsx"

In [4]:
# Loading sheets
bus = pd.read_excel(file_path, sheet_name="Bus", header=3)
line = pd.read_excel(file_path, sheet_name="Line", header=3)
gen = pd.read_excel(file_path, sheet_name="Generator", header=3)
load = pd.read_excel(file_path, sheet_name="Load", header=3)
hvdc = pd.read_excel(file_path, sheet_name="HVDC", header=3)
transformer2 = pd.read_excel(file_path, sheet_name="Transformer2", header=3)
transformer3 = pd.read_excel(file_path, sheet_name="Transformer3", header=3)

for df in [bus, line, gen, load, hvdc, transformer2, transformer3]:
    df.columns = [str(c).strip() for c in df.columns]


# This is our main cleaning function that turns our raw data into an unaggregated graph, that is used for the scenario generation
g_unaggregated = cleaning.main_clean(bus, line, gen, load, hvdc, transformer2, transformer3)
edges = pd.read_excel('danish_grid_graph_ready.xlsx', sheet_name='edges')
nodes = pd.read_excel('danish_grid_graph_ready.xlsx', sheet_name='nodes')

NameError: name 'pd' is not defined

In [ ]:
cleaning.print_graph(cleaning.aggregate_graph(g_unaggregated, edges))

In [ ]:
cleaning.print_graph_coordinates(cleaning.aggregate_graph(g_unaggregated, edges))

## Scenario-conditioned attack simulations

Only balanced scenarios (supply ≈ demand after rebalancing) are valid baselines for attack evaluation.
If the grid already has unmet demand before an attack, the simulation results cannot isolate the attack's impact.

Only balanced scenarios are considered valid:

$$
\text { gap }=\mid \text { supply }- \text { demand } \mid
$$


A scenario is accepted if:

$$
\frac{\text { gap }}{\text { demand }} \leq 1 \%
$$


This ensures that any imbalance observed during simulations is caused only by the simulated failure, not by initial conditions.

In [ ]:
base_scenario = g_unaggregated

In [ ]:
from scenarios import SCENARIOS, apply_scenario_mean

# ── Check balance for every scenario ─────────────────────────────────────────
BALANCE_TOL_PCT = 1.0   # allow up to 1 % residual gap

balance_rows = []
for sid, scenario in SCENARIOS.items():
    G_sc = apply_scenario_mean(base_scenario, scenario)
    total_supply = sum(d.get('supply', 0.0) for _, d in G_sc.nodes(data=True))
    total_demand = sum(d.get('demand', 0.0) for _, d in G_sc.nodes(data=True))
    gap          = abs(total_supply - total_demand)
    gap_pct      = gap / max(total_demand, 1.0) * 100
    balance_rows.append({
        'scenario_id'     : sid,
        'label'           : scenario['label'],
        'total_supply_MW' : round(total_supply, 1),
        'total_demand_MW' : round(total_demand, 1),
        'gap_MW'          : round(gap, 1),
        'gap_pct'         : round(gap_pct, 3),
        'balanced'        : gap_pct <= BALANCE_TOL_PCT,
    })

balance_df = pd.DataFrame(balance_rows)
balance_df

Getting all single removals of nodes and edges for each scenario

For each valid scenario, we simulate single-component failures:
- Node removals (substation failures)
- Edge removals (line or transformer outages)

In [ ]:
balanced_ids = balance_df.loc[balance_df['balanced'], 'scenario_id'].tolist()
skipped_ids  = balance_df.loc[~balance_df['balanced'], 'scenario_id'].tolist()

print(f"Balanced ({len(balanced_ids)}): {balanced_ids}")
if skipped_ids:
    print(f"Skipped  ({len(skipped_ids)}): {skipped_ids}")

# ── Run single-removal attack simulation for each balanced scenario ──────────
scenario_results_list = []
for sid in balanced_ids:
    scenario = SCENARIOS[sid]
    G_sc = apply_scenario_mean(base_scenario, scenario)
    G_sc.name = sid
    print(f"\n→ {sid}: {scenario['label']}")
    df = Evaluation.simulation_all_single_removals(cleaning.aggregate_graph(G_sc, edges))
    df.insert(0, 'scenario_id',    sid)
    df.insert(1, 'scenario_label', scenario['label'])
    scenario_results_list.append(df)

scenario_results = pd.concat(scenario_results_list, ignore_index=True)
scenario_results.to_excel("scenario_results.xlsx", index=False)
print(f"\nSaved {len(scenario_results)} rows to scenario_results.xlsx")

In [ ]:
scenario_results.sort_values('total_unmet_demand_MW', ascending=False).head(20)

In [ ]:
print(balanced_ids[1:])
print(balanced_ids[0])

## Optimization Model

Each failure scenario is evaluated using a linear optimization model based on power flow and load shedding


Decision Variables
- $t_{u v}$ : power flow on each line
- $s_{\text {pos }}$ : oversupply (curtailment)
- $s_{n e g}$ : unmet demand (load shedding)
- $\delta_{\text {add }}, \delta_{\text {remove: }}$ : generation adjustments

Objective Function
Minimize total system imbalance:

$$
\min \sum_n\left(s_{p o s, n}+s_{n e g, n}\right)
$$

Constraints
1. Flow balance at each node:

$$
\text { supply }+\delta_{a d d}-\delta_{\text {remove }}-\text { demand }+ \text { inflow }- \text { outflow }-s_{\text {pos }}+s_{n e g}=0
$$

2. Transmission capacity:

$$
-C_{u v} \leq t_{u v} \leq C_{u v}
$$

3. Generation flexibility limits


## Performance Metrics

Each attack is evaluated using:

- Total unmet demand (MW) → primary indicator of failure severity
- Total oversupply (MW) → stranded generation
- Number of connected components → grid fragmentation

## Assessment

Each node/edge is assigned a criticality score based on:

system imbalance after removal
structural impact on connectivity

Initially, results are ranked directly by unmet demand.

Later, a normalized weighted score is used:
$$
\text { score }=0.7 \cdot \text { unmet }_{\text {norm }}+0.2 \cdot \text { oversupply }_{\text {norm }}+0.1 \cdot \text { groups }_{\text {norm }}
$$
This ensures comparability across metrics.


## GNN: Now we'll train a GNN on this to come up with good predictions

In [ ]:
from GNN import train_gnn, predict_criticality

scenario_graphs = {}
for sid in balanced_ids[1:]:
    scenario = SCENARIOS[sid]
    G_sc = apply_scenario_mean(base_scenario, scenario)
    scenario_graphs[sid] = cleaning.aggregate_graph(G_sc, edges)

model, t_mean, t_std = train_gnn(scenario_graphs, scenario_results, epochs=500)

#print(scenario_graphs)

In [ ]:
sid = balanced_ids[0]
scenario = SCENARIOS[sid]
G_sc = apply_scenario_mean(base_scenario, scenario)
G_test = cleaning.aggregate_graph(G_sc, edges)


preds = predict_criticality(model, G_test, t_mean, t_std)

# Actual results for S03 from simulation
actual = (scenario_results[scenario_results["scenario_id"] == sid]
          [["removals", "name", "attack", "total_unmet_demand_MW"]]
          .copy())
actual["removals"] = actual["removals"].astype(str)
preds["removal"]   = preds["removal"].astype(str)

comparison = (preds
    .merge(actual, left_on="removal", right_on="removals", how="left")
    .rename(columns={"total_unmet_demand_MW": "actual_unmet_MW"})
    [["attack_x", "removal", "name_x", "predicted_unmet_MW", "actual_unmet_MW"]]
    .rename(columns={"attack_x": "attack", "name_x": "name"})
)
comparison.head(20)

Checking the supply of the scenarios

In [ ]:
import matplotlib.pyplot as plt

# Build supply-by-source table
source_rows = []
for sid, G_sc in scenario_graphs.items():
    for _, data in G_sc.nodes(data=True):
        supply = data.get("supply", 0.0)
        if supply <= 0:
            continue
        for src in str(data.get("source", "unknown")).split(", "):
            source_rows.append({"scenario_id": sid, "source": src.strip(), "supply_MW": supply})

supply_by_source = (
    pd.DataFrame(source_rows)
    .groupby(["scenario_id", "source"])["supply_MW"]
    .sum()
    .unstack(fill_value=0)
    .round(1)
)

# Stacked bar chart
ax = supply_by_source.plot(
    kind="bar", stacked=True, figsize=(14, 6),
    colormap="tab10", width=0.7
)
ax.set_title("Total Supply by Source per Scenario (MW)")
ax.set_xlabel("")
ax.set_ylabel("Supply (MW)")
ax.set_xticklabels([s.replace("_", " ") for s in supply_by_source.index], rotation=30, ha="right")
ax.legend(title="Source", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()